# Week 1 Thursday — Assemble the SageMaker Pipeline

Chains: preprocessing -> training -> evaluation -> conditional model registration.

## Setup — role, processor, estimator (re-defined here since this is a fresh notebook/kernel)

In [ ]:
import sagemaker
print(sagemaker.__version__)  # should print 2.257.5

from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.xgboost import XGBoost

role = "arn:aws:iam::666258711441:role/service-role/AmazonSageMaker-ExecutionRole-20260722T143599"
BUCKET = "beant-mlops-portfolio-666258711441"

processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type="ml.t3.medium",
    instance_count=1,
)

estimator = XGBoost(
    entry_point="train.py",
    framework_version="1.7-1",
    instance_type="ml.m5.large",
    instance_count=1,
    role=role,
    hyperparameters={
        "max_depth": 5,
        "eta": 0.173,
        "num_round": 100,
        "objective": "binary:logistic",
        "scale_pos_weight": 2.77,
    },
)

## Step 1 — Pipeline parameters

In [ ]:
from sagemaker.workflow.parameters import ParameterString, ParameterFloat

processing_instance_type = ParameterString(name="ProcessingInstanceType", default_value="ml.t3.medium")
training_instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.large")
f1_threshold = ParameterFloat(name="F1Threshold", default_value=0.60)

## Step 2a — ProcessingStep (reuses `preprocessing.py`)

In [ ]:
from sagemaker.workflow.steps import ProcessingStep

process_step = ProcessingStep(
    name="PreprocessChurnData",
    processor=processor,
    inputs=[
        ProcessingInput(
            source=f"s3://{BUCKET}/raw/",
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/output/train"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/output/test"),
    ],
    code="preprocessing.py",
)

## Step 2b — TrainingStep (reuses `train.py`)

In [ ]:
from sagemaker.workflow.steps import TrainingStep
from sagemaker.inputs import TrainingInput

train_step = TrainingStep(
    name="TrainChurnModel",
    estimator=estimator,
    inputs={
        "train": TrainingInput(
            s3_data=process_step.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri
        ),
        "validation": TrainingInput(
            s3_data=process_step.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri
        ),
    },
)

## Step 2c — Evaluation ProcessingStep (reuses `evaluate.py`)

Uses a `ScriptProcessor` on the XGBoost container image (not `SKLearnProcessor`) since `evaluate.py` needs `xgboost` installed to load the model.

In [ ]:
from sagemaker.processing import ScriptProcessor
from sagemaker.workflow.properties import PropertyFile
from sagemaker.image_uris import retrieve

xgb_image = retrieve("xgboost", region="ca-central-1", version="1.7-1")

eval_processor = ScriptProcessor(
    image_uri=xgb_image,
    command=["python3"],
    instance_type="ml.t3.medium",
    instance_count=1,
    role=role,
)

evaluation_report = PropertyFile(name="EvaluationReport", output_name="evaluation", path="evaluation.json")

eval_step = ProcessingStep(
    name="EvaluateChurnModel",
    processor=eval_processor,
    inputs=[
        ProcessingInput(
            source=train_step.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=process_step.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation")],
    code="evaluate.py",
    property_files=[evaluation_report],
)

## Step 3 — ConditionStep + RegisterModel

F1 threshold set to 0.60 (Monday's acceptance gate) — the `scale_pos_weight=2.77` run cleared this at F1=0.6211, so this pipeline should register the model.

In [ ]:
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThan
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.step_collections import RegisterModel

register_step = RegisterModel(
    name="RegisterChurnModel",
    estimator=estimator,
    model_data=train_step.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium"],
    transform_instances=["ml.m5.large"],
    model_package_group_name="churn-xgboost-models",
    approval_status="PendingManualApproval",
)

condition = ConditionGreaterThan(
    left=JsonGet(
        step_name=eval_step.name,
        property_file=evaluation_report,
        json_path="classification_metrics.f1.value",
    ),
    right=f1_threshold,
)

condition_step = ConditionStep(
    name="CheckF1Threshold",
    conditions=[condition],
    if_steps=[register_step],
    else_steps=[],
)

## Step 4 — Assemble and upsert

Note: `pipeline.start()` is Friday's step, not run here.

In [ ]:
from sagemaker.workflow.pipeline import Pipeline

pipeline = Pipeline(
    name="churn-detection-pipeline",
    parameters=[processing_instance_type, training_instance_type, f1_threshold],
    steps=[process_step, train_step, eval_step, condition_step],
)
pipeline.upsert(role_arn=role)